# ODA-LAB SET1R — Drive API + spinner

```yaml
skill: olivia-dev-alpha
job: SET1R
envelope: tui
```

**Runtime → Run all.** Second click is Google auth so we can fetch by file id.
Fixes the nine gdown misses. LIV_PANE will EXISTS-skip if already loose.
Dest is `MyDrive/12_ODA-LAB-NOTEBOOKS/extracts/SET1R-<stamp>/` — created at Drive root.


In [ ]:
from google.colab import drive
print('=== SET1R mount ===')
drive.mount('/content/drive', force_remount=False)
print('mounted')


In [ ]:

import io, json, sys, time, zipfile, tarfile, traceback
from pathlib import Path
from datetime import datetime, timezone

PACKS = [["STORYBOARD_RECON_20260910.zip", "1dYjyx6S1Tp3pWvPC6hchxe6OV5gQ69DX", "zip"], ["LIV_PANE_ARCHIVE_20260910.zip", "1518UAB_yra8VEvfTiRef-fEnZrfjZ5pB", "zip"], ["plate-pack-normalized-20260910.zip", "1G1pHWZuIRVEt9XIanZ6jmjeQAXJ3G7yE", "zip"], ["EMIT-20260910.zip", "1ZokwsXIfogBZfL2sP5du4n_q2Pnn9M5H", "zip"], ["ANIME-COPPER-20260910.zip", "1b7xEj1Wft8DJQzWH4GzQ_4TICnZNQM0i", "zip"], ["GRID-BLONDE-20260910.zip", "1yVCuq80O-eMI6E9OYJ_iXZfzCL05K0tG", "zip"], ["cand-reel-133985.zip", "12NkSa8VC_Qe13eqisQjuZiq_F3pefn8Y", "zip"], ["IP-WQ-161-166.zip", "18LTJ0HrF7DMuNpB0W2JMzZ1vaj1Rjd4T", "zip"], ["IP-WQ-161-172.zip", "1t5Zal_pLKz5fwSANPnRjMy6YSbdMDl3W", "zip"], ["OLIVIA_ORCA_SESSION_20260901.zip", "1uvLj8Gs0hluOPVVUukLNUqiLZYr9ppqa", "zip"]]
CAP = 500 * 1024 * 1024
SPIN = "|/-\\"
ROOT = Path("/content/drive/MyDrive")
SHELF = ROOT / "12_ODA-LAB-NOTEBOOKS"
SHELF.mkdir(parents=True, exist_ok=True)
STAMP = time.strftime("%Y%m%d-%H%M")
DEST = SHELF / "extracts" / ("SET1R-" + STAMP)
DEST.mkdir(parents=True, exist_ok=True)
LOG = DEST / "EXTRACT.LOG.md"
REC = DEST / "EXTRACT.RECEIPT.md"

def log(msg):
    line = time.strftime("%H:%M:%S") + " " + msg
    print(line, flush=True)
    with LOG.open("a") as f:
        f.write(line + "\n")

def bar(pct, name, spin_i, rc=""):
    pct = max(0, min(100, int(pct)))
    filled = pct // 5
    body = "=" * filled + (">" if filled < 20 else "")
    body = body[:20].ljust(20)
    ch = SPIN[spin_i % 4]
    tail = (" rc=" + str(rc)) if rc != "" else ""
    print(f"\r  [{body}] {pct:3d}% {ch} {name}{tail}   ", end="", flush=True)

print("┌" + "─"*62 + "┐")
print("│ ODA LAB SET1R   format-bible tui   C-64 dash                  │")
print("│ dest " + str(DEST)[:55].ljust(55) + "│")
print("└" + "─"*62 + "┘")
log("start dest=" + str(DEST))

from google.colab import auth
auth.authenticate_user()
import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
creds, _ = google.auth.default()
svc = build("drive", "v3", credentials=creds)
log("drive v3 ok")

def pull(fid, dest_path):
    request = svc.files().get_media(fileId=fid)
    fh = io.FileIO(dest_path, "wb")
    dl = MediaIoBaseDownload(fh, request, chunksize=1024*1024)
    done = False
    i = 0
    while not done:
        status, done = dl.next_chunk()
        i += 1
        pct = int(status.progress() * 100) if status else 0
        bar(pct, dest_path.name, i)
    fh.close()
    print()
    return Path(dest_path).stat().st_size

rows = []
fail = 0
ok = 0
for name, fid, kind in PACKS:
    print()
    log("FILE " + name + " id=" + fid)
    out = DEST / Path(name).stem
    if out.exists() and any(out.iterdir()):
        cnt = sum(1 for p in out.rglob("*") if p.is_file())
        log("EXISTS files=" + str(cnt) + " rc=0")
        rows.append("- EXISTS " + name + " files=" + str(cnt) + " rc=0")
        ok += 1
        continue
    local = Path("/content") / name
    try:
        sz = pull(fid, local)
    except Exception as e:
        log("DOWNLOAD FAIL rc=2 " + repr(e))
        traceback.print_exc()
        rows.append("- FAIL download " + name + " rc=2 " + str(e)[:120])
        fail += 1
        continue
    if sz > CAP:
        log("SKIP cap rc=4 bytes=" + str(sz))
        rows.append("- SKIP cap " + name + " rc=4")
        fail += 1
        continue
    out.mkdir(parents=True, exist_ok=True)
    try:
        if kind == "zip":
            with zipfile.ZipFile(local) as z:
                z.extractall(out)
                nlist = z.namelist()[:5]
        else:
            with tarfile.open(local, "r:*") as t:
                t.extractall(out)
                nlist = t.getnames()[:5]
        cnt = sum(1 for p in out.rglob("*") if p.is_file())
        log("OK bytes=" + str(sz) + " files=" + str(cnt) + " rc=0")
        rows.append("- OK " + name + " bytes=" + str(sz) + " files=" + str(cnt) + " rc=0 first=" + str(nlist))
        ok += 1
        bar(100, name, 0, rc=0)
        print()
    except Exception as e:
        log("EXTRACT FAIL rc=3 " + repr(e))
        rows.append("- FAIL extract " + name + " rc=3 " + str(e)[:120])
        fail += 1

final = 0 if fail == 0 else 1
front = """---
skill: olivia-dev-alpha
module: colab-launcher
job: SET1R
envelope: tui
clock: %s
ok: %s
fail: %s
exit: %s
dest: %s
---
""" % (datetime.now(timezone.utc).isoformat(), ok, fail, final, DEST)
REC.write_text(front + "\n".join(rows) + "\n")
log("done exit=" + str(final) + " ok=" + str(ok) + " fail=" + str(fail))
print()
print("RECEIPT", REC)
print("LOG", LOG)
print("EXIT", final)
